# PCA Analysis — Student Dropout Dataset

**Goal:** Determine the optimal number of PCA components that capture ≥ 95% of total variance in the 36-feature dataset. Findings feed directly into the training pipeline.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT       = Path("../")
DATA_PATH  = ROOT / "data" / "Predict Student Dropout and Academic Success.csv"
PLOTS_DIR  = ROOT / "results" / "plots"
RESULTS_DIR = ROOT / "results"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries loaded. Data path:", DATA_PATH.resolve())

In [ ]:
# ── Load & prepare data ────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)

# Binary-encode target: Dropout=1, everything else (Graduate / Enrolled)=0
df['Target'] = df['Target'].apply(lambda x: 1 if x == 'Dropout' else 0)

X = df.drop('Target', axis=1)
y = df['Target']

print(f"Dataset shape : {df.shape}")
print(f"Feature count : {X.shape[1]}")
print(f"Target counts :\n{y.value_counts().rename({0:'Non-dropout', 1:'Dropout'})}")
print("\nColumns:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# ── Train/test split (same split as modelTraining.ipynb) ──────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

In [ ]:
# ── Scale → full PCA (all 36 components) to analyse variance ──────────────────
scaler_pca = StandardScaler()
X_train_scaled = scaler_pca.fit_transform(X_train)
X_test_scaled  = scaler_pca.transform(X_test)

pca_full = PCA(n_components=X_train.shape[1], random_state=42)
pca_full.fit(X_train_scaled)

explained            = pca_full.explained_variance_ratio_
cumulative           = np.cumsum(explained)
component_numbers    = np.arange(1, len(explained) + 1)

print("Component | Individual % | Cumulative %")
print("-" * 42)
for i, (ind, cum) in enumerate(zip(explained, cumulative), 1):
    marker = " ← 95%" if cum >= 0.95 and (i == 1 or cumulative[i-2] < 0.95) else ""
    marker90 = " ← 90%" if cum >= 0.90 and (i == 1 or cumulative[i-2] < 0.90) else ""
    print(f"  PC{i:2d}     |   {ind*100:6.2f}%    |   {cum*100:6.2f}%  {marker}{marker90}")

In [ ]:
# ── Determine optimal n_components ────────────────────────────────────────────
THRESHOLD_90 = 0.90
THRESHOLD_95 = 0.95

n_90 = int(np.argmax(cumulative >= THRESHOLD_90)) + 1
n_95 = int(np.argmax(cumulative >= THRESHOLD_95)) + 1

# Choose 95% threshold as the project target
OPTIMAL_N = n_95

print(f"Components for ≥ 90% variance : {n_90}")
print(f"Components for ≥ 95% variance : {n_95}")
print(f"\n✅ Chosen n_components = {OPTIMAL_N}  (captures {cumulative[OPTIMAL_N-1]*100:.2f}% variance)")

In [ ]:
# ── Plot explained variance ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0b0f1a')

ACCENT  = '#6366f1'
GREEN   = '#22c55e'
AMBER   = '#f59e0b'
MUTED   = '#94a3b8'
BG_CARD = '#111827'

for ax in axes:
    ax.set_facecolor(BG_CARD)
    ax.tick_params(colors=MUTED)
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    ax.title.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1f2937')

# ─ Left: individual explained variance (bar) ─
ax1 = axes[0]
colors = [ACCENT if i < OPTIMAL_N else '#374151' for i in range(len(explained))]
ax1.bar(component_numbers, explained * 100, color=colors, width=0.7, zorder=3)
ax1.set_xlabel('Principal Component', fontsize=10)
ax1.set_ylabel('Explained Variance (%)', fontsize=10)
ax1.set_title('Individual Explained Variance', fontsize=12, fontweight='bold', pad=12)
ax1.grid(axis='y', color='#1f2937', linewidth=0.8, zorder=0)
ax1.set_xticks(component_numbers[::3])

# ─ Right: cumulative explained variance (line) ─
ax2 = axes[1]
ax2.plot(component_numbers, cumulative * 100,
         color=ACCENT, linewidth=2.5, marker='o', markersize=4, label='Cumulative variance')

# Threshold lines
ax2.axhline(90, color=AMBER, linestyle='--', linewidth=1.4, label='90% threshold')
ax2.axhline(95, color=GREEN, linestyle='--', linewidth=1.4, label='95% threshold')

# Mark chosen component
ax2.axvline(OPTIMAL_N, color=GREEN, linestyle=':', linewidth=1.4)
ax2.scatter([OPTIMAL_N], [cumulative[OPTIMAL_N-1]*100],
            color=GREEN, s=100, zorder=5, label=f'Chosen: PC{OPTIMAL_N} ({cumulative[OPTIMAL_N-1]*100:.1f}%)')

ax2.set_xlabel('Number of Components', fontsize=10)
ax2.set_ylabel('Cumulative Variance (%)', fontsize=10)
ax2.set_title('Cumulative Explained Variance', fontsize=12, fontweight='bold', pad=12)
ax2.set_ylim(0, 103)
ax2.grid(color='#1f2937', linewidth=0.8, zorder=0)
ax2.legend(fontsize=9, facecolor='#1f2937', edgecolor='#374151', labelcolor='white')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout(pad=2.5)

plot_path = PLOTS_DIR / 'pca_variance.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f"\nPlot saved → {plot_path.resolve()}")

In [ ]:
# ── Save findings ──────────────────────────────────────────────────────────────
findings = f"""PCA Analysis Findings
=====================
Dataset            : Predict Student Dropout and Academic Success
Total features     : {X.shape[1]}
Training samples   : {X_train.shape[0]}

Variance thresholds
-------------------
n_components for ≥ 90% variance : {n_90}  ({cumulative[n_90-1]*100:.2f}%)
n_components for ≥ 95% variance : {n_95}  ({cumulative[n_95-1]*100:.2f}%)

SELECTED n_components = {OPTIMAL_N}
Variance captured     = {cumulative[OPTIMAL_N-1]*100:.4f}%
Dimensionality ratio  = {OPTIMAL_N}/{X.shape[1]} = {OPTIMAL_N/X.shape[1]:.2%}

Top 10 components (cumulative variance)
---------------------------------------
"""

for i in range(min(10, len(cumulative))):
    findings += f"  PC{i+1:2d} : {explained[i]*100:6.3f}%  (cumulative: {cumulative[i]*100:.3f}%)\n"

findings_path = RESULTS_DIR / 'pca_findings.txt'
findings_path.write_text(findings)

print(findings)
print(f"Findings saved → {findings_path.resolve()}")

---
## Summary

The analysis above determines `OPTIMAL_N` (the chosen `n_components`) based on the 95% cumulative explained variance threshold. This value is used directly in `train_pca_pipeline.py` to build the sklearn Pipeline.

**Next step:** Run `notebooks/train_pca_pipeline.py` to train and save `models/model_pca_pipeline.pkl`.